In [1]:
import spatialdata as sd
from spatialdata_io import xenium
import scanpy as sc
import pandas as pd
import geopandas as gpd

/home/xilab/software/miniconda-envs/python-xenium/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/xilab/software/miniconda-envs/python-xenium/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [2]:
zarr_path='/home/xilab/jiangdx/Xenium-spatial/mouse-embryo/featal-SI/scanpy-for-spatial-analysis/Xenium.zarr'
sdata = sd.read_zarr(zarr_path)
sdata

/home/xilab/software/miniconda-envs/python-xenium/lib/python3.12/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/home/xilab/software/miniconda-envs/python-xenium/lib/python3.12/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/home/xilab/software/miniconda-envs/python-xenium/lib/python3.12/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/home/xilab/software/miniconda-envs/python-xenium/lib/python3.12/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/home/xilab/software/miniconda-envs/python-xenium/lib/python3.12/site-packages/zarr/creation.py:

SpatialData object, with associated Zarr store: /data2/shared_data_backup/jiangdx/Xenium-spatial/mouse-embryo/featal-SI/scanpy-for-spatial-analysis/Xenium.zarr
├── Images
│     └── 'morphology_focus': DataTree[cyx] (5, 105314, 48359), (5, 52657, 24179), (5, 26328, 12089), (5, 13164, 6044), (5, 6582, 3022)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (105314, 48359), (52657, 24179), (26328, 12089), (13164, 6044), (6582, 3022)
│     └── 'nucleus_labels': DataTree[yx] (105314, 48359), (52657, 24179), (26328, 12089), (13164, 6044), (6582, 3022)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (695471, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (695471, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (657330, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (695471, 5105)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus

# GeoPandas
> [GeoPandas Chinese](https://blog.csdn.net/qazwsxpy/article/details/123347777)
> 
> [GeoPandas API](https://blog.csdn.net/qq_35240689/article/details/143875707)

In [3]:
gdf = sdata.shapes['cell_boundaries']
# gdf["geometry"].plot()

# extend cell boundary by 1 um
gdf['buffered'] = gdf.buffer(1)

# # calculate centorid for each cell, later used for clip center cell region
# gdf['centroid'] = gdf.centroid
# # extend the centorid by 100um to clip gdf and find neighbours for center cell
# gdf["buffered_centroid"] = gdf["centroid"].buffer(100)
# # remove invalid geometry
# gdf = gdf.loc[gdf.buffered_centroid.is_valid]

In [4]:
# clip_area = gdf.loc[['cppgblpl-1'], "buffered_centroid"]
# gdf_clipped = gpd.clip(gdf, clip_area)
# print(gdf_clipped.shape)
# gdf_clipped

In [5]:
# gdf['buffered'] = gdf.buffer(1)
# # plot cells
# selected_cell = gdf.loc[['cppgfkhl-1'], "geometry"]
# selected_cell.plot()
# selected_cell_buffered = gdf.loc[['cppgfkhl-1'], "buffered"]
# selected_cell_buffered.plot()
# selected_cells_buffered = gdf.loc[['cppgfkhl-1','cppgblpl-1'], "buffered"]
# selected_cells_buffered.plot()
# selected_cells= gdf.loc[['cppgfkhl-1','cppgblpl-1'], "geometry"]
# selected_cells.plot()

# only keep featal cells to accelerate run speed

In [45]:
featal_cell_meta = pd.read_csv("/home/xilab/jiangdx/Xenium-spatial/mouse-embryo/featal-SI/scanpy-for-spatial-analysis/all-cell-types/metadata-fro-scanpy.csv", index_col=0)
featal_cell_meta.head()


Index(['affjoobf-1', 'affjooep-1', 'affkdbkd-1', 'affkfcec-1', 'affkmjan-1',
       'afflaebo-1', 'afflafpm-1', 'afflefip-1', 'affljcif-1', 'affljkkj-1',
       ...
       'oigaadhh-1', 'oigaddma-1', 'oigaffoi-1', 'oigagdci-1', 'oigagfbm-1',
       'oigahppi-1', 'oigamclp-1', 'oigamnjd-1', 'oigbdmlb-1', 'oigbikph-1'],
      dtype='object', length=17268)

In [7]:
def find_neighbours(gdf, center_cell):
    # clip gdf base on a circle area with centorid from center cell.
    # clip_area = gdf.loc[[center_cell], "buffered_centroid"]
    # gdf = gpd.clip(gdf, clip_area)
    center_cell_geometry = gdf.loc[[center_cell], "geometry"]
    #> https://stackoverflow.com/questions/68386172/geopandas-known-intersection-returns-false
    center_cell_intersection=gdf['buffered'].intersects(center_cell_geometry.iloc[0])
    center_cell_neighbours=center_cell_intersection.index[center_cell_intersection].tolist()
    return center_cell_neighbours
    
cell_meta=pd.read_csv("/home/xilab/jiangdx/Xenium-spatial/mouse-embryo/featal-SI/scanpy-for-spatial-analysis/only-epi/epi-metadata-fro-scanpy.csv", index_col=0)
all_cell_types=cell_meta.RCTD_2_first_type.unique().tolist()
all_days=cell_meta.day.unique().tolist()

# normalize by center cell counts
divide the accumulated neighbour cell counts by the center numbers

In [48]:
for day in all_days:
    print("Processing day - " + day + "...")
    # subset cell meta
    cell_meta_sub = cell_meta[cell_meta.day == day]
    #print(cell_meta_sub.shape)
    #print(cell_meta_sub.RCTD_2_first_type.value_counts())
    # get all cell annotations
    cells_dict = cell_meta_sub.RCTD_2_first_type.to_dict()
    # save nearest neighbours    
    first_neighbours_df=pd.DataFrame(index=all_cell_types)
    # save secondary neigbhours
    second_neighbours_df=pd.DataFrame(index=all_cell_types)
    # subset gdf to accelerate analysis
    cells_day=featal_cell_meta.index[featal_cell_meta.day == day]
    gdf_sub=gdf[gdf.index.isin(cells_day)]
    for celltype in all_cell_types:
        # get all cell ids (index) for center cells
        center_cells = cell_meta_sub.index[cell_meta_sub.RCTD_2_first_type == celltype].tolist()
        # to save cell neighbours static results to a dict
        cell_neighbours_dict = dict()
        # find cell neighbours for each center cell
        for center_cell in center_cells:
            # find nearest neighbours
            center_cell_neighbours=find_neighbours(gdf=gdf_sub, center_cell=center_cell)
            center_cell_neighbours.remove(center_cell)
            # find secondary neighbours, attention: not remove duplicated nearest neighbours!
            secondary_neighours=[]
            for center_cell_neighbour in center_cell_neighbours:
                secondary_neighours = secondary_neighours + (find_neighbours(gdf=gdf_sub, center_cell=center_cell_neighbour))
            # remove duplicated secondary neighbours
            secondary_neighours = list(set(secondary_neighours))
            # remvoe center cell and nearest neighbours
            centers_all = center_cell + center_cell_neighbour
            for i in centers_all:
                if i in secondary_neighours:
                    secondary_neighours.remove(i)
            cell_neighbours_dict[center_cell] = [center_cell_neighbours, secondary_neighours]
        # accumulate the cell neighbours for all center cells
        nearest_neighbours=[]
        secondary_neighours=[]
        nearest_neighbour_counts=0
        secondary_neighour_counts=0
        for center_cell, neighbours in cell_neighbours_dict.items():
            nearest_neighbour_counts=nearest_neighbour_counts + len(neighbours[0])
            for i in neighbours[0]:
                if i in cells_dict.keys():
                    nearest_neighbours = nearest_neighbours + [cells_dict[i]]
            secondary_neighour_counts=secondary_neighour_counts + len(neighbours[1])
            for i in neighbours[1]:
                if i in cells_dict.keys():
                    secondary_neighours = secondary_neighours + [cells_dict[i]]
        # format the results for nearest neighbours
        data = dict((i, nearest_neighbours.count(i)) for i in all_cell_types)
        res = pd.DataFrame.from_dict(data, orient='index',columns=[celltype])
        # normalize neighbour counts by center cell numbers
        if len(center_cells) != 0:
            res = res.div(len(center_cells))
        first_neighbours_df = pd.concat([first_neighbours_df, res], axis=1)
         # format the results for secondary neighbours
        data = dict((i, secondary_neighours.count(i)) for i in all_cell_types)
        res = pd.DataFrame.from_dict(data, orient='index', columns=[celltype])
        # normalize neighbour counts by center cell numbers
        if len(center_cells) != 0:
            res = res.div(len(center_cells))
        second_neighbours_df = pd.concat([second_neighbours_df, res], axis=1)
        # print(pd.DataFrame.from_dict(dict((i, nearest_neighbours.count(i)) for i in all_cell_types)))
        # print(pd.DataFrame.from_dict(dict((i, secondary_neighours.count(i)) for i in all_cell_types)))
    first_neighbours_df.to_csv("outputs/" + day + "-first-mean.csv")
    second_neighbours_df.to_csv("outputs/" + day + "-second-mean.csv")


Processing day - E19.5_SI...
Processing day - E14.5_SI...
Processing day - E15.5_SI...
Processing day - E13.5_SI...
Processing day - E12.5_SI...


# neighbour cell type percentage
calculate the percentage for each accumulated cell type.

In [51]:
for day in all_days:
    print("Processing day - " + day + "...")
    # subset cell meta
    cell_meta_sub = cell_meta[cell_meta.day == day]
    #print(cell_meta_sub.shape)
    #print(cell_meta_sub.RCTD_2_first_type.value_counts())
    # get all cell annotations
    cells_dict = cell_meta_sub.RCTD_2_first_type.to_dict()
    # save nearest neighbours    
    first_neighbours_df=pd.DataFrame(index=all_cell_types)
    # save secondary neigbhours
    second_neighbours_df=pd.DataFrame(index=all_cell_types)
    # subset gdf to accelerate analysis
    cells_day=featal_cell_meta.index[featal_cell_meta.day == day]
    gdf_sub=gdf[gdf.index.isin(cells_day)]
    for celltype in all_cell_types:
        # get all cell ids (index) for center cells
        center_cells = cell_meta_sub.index[cell_meta_sub.RCTD_2_first_type == celltype].tolist()
        # to save cell neighbours static results to a dict
        cell_neighbours_dict = dict()
        # find cell neighbours for each center cell
        for center_cell in center_cells:
            # find nearest neighbours
            center_cell_neighbours=find_neighbours(gdf=gdf_sub, center_cell=center_cell)
            center_cell_neighbours.remove(center_cell)
            # find secondary neighbours, attention: not remove duplicated nearest neighbours!
            secondary_neighours=[]
            for center_cell_neighbour in center_cell_neighbours:
                secondary_neighours = secondary_neighours + (find_neighbours(gdf=gdf_sub, center_cell=center_cell_neighbour))
            # remove duplicated secondary neigubours
            secondary_neighours = list(set(secondary_neighours))
            # remvoe center cell and nearest neighbours
            centers_all = center_cell + center_cell_neighbour
            for i in centers_all:
                if i in secondary_neighours:
                    secondary_neighours.remove(i)
            cell_neighbours_dict[center_cell] = [center_cell_neighbours, secondary_neighours]
        # accumulate the cell neighbours for all center cells
        nearest_neighbours=[]
        secondary_neighours=[]
        nearest_neighbour_counts=0
        secondary_neighour_counts=0
        for center_cell, neighbours in cell_neighbours_dict.items():
            nearest_neighbour_counts=nearest_neighbour_counts + len(neighbours[0])
            for i in neighbours[0]:
                if i in cells_dict.keys():
                    nearest_neighbours = nearest_neighbours + [cells_dict[i]]
            secondary_neighour_counts=secondary_neighour_counts + len(neighbours[1])
            for i in neighbours[1]:
                if i in cells_dict.keys():
                    secondary_neighours = secondary_neighours + [cells_dict[i]] 
        # format the results for nearest neighbours
        data = dict((i, nearest_neighbours.count(i)) for i in all_cell_types)
        res = pd.DataFrame.from_dict(data, orient='index',columns=[celltype])
        # normalize neighbour counts by total neighbours
        if nearest_neighbour_counts != 0:
            res = res.div(nearest_neighbour_counts) * 100
        first_neighbours_df = pd.concat([first_neighbours_df, res], axis=1)
         # format the results for secondary neighbours
        data = dict((i, secondary_neighours.count(i)) for i in all_cell_types)
        res = pd.DataFrame.from_dict(data, orient='index', columns=[celltype])
        # normalize neighbour counts by total neighbours
        if secondary_neighour_counts != 0:
            res = res.div(secondary_neighour_counts) * 100
        second_neighbours_df = pd.concat([second_neighbours_df, res], axis=1)
        # print(pd.DataFrame.from_dict(dict((i, nearest_neighbours.count(i)) for i in all_cell_types)))
        # print(pd.DataFrame.from_dict(dict((i, secondary_neighours.count(i)) for i in all_cell_types)))
    first_neighbours_df.to_csv("outputs/" + day + "-first-percentage.csv")
    second_neighbours_df.to_csv("outputs/" + day + "-second-percentage.csv")

Processing day - E19.5_SI...
Processing day - E14.5_SI...
Processing day - E15.5_SI...
Processing day - E13.5_SI...
Processing day - E12.5_SI...
